# E1/E2: decomposition quality and routing accuracy

Thin Kaggle runner for `experiments/e1_e2.py`. This notebook does **not** train or fine-tune anything.

**Attach one COCO-Stuff dataset containing:**

- `images/train2017` and `images/val2017`
- matching `stuffthingmaps/train2017` and `stuffthingmaps/val2017` PNGs
- the official `labels.txt`

A SAM ViT-B checkpoint is optional; if it is not attached, `GroundedSAM` downloads it on first use (Kaggle internet must be enabled).

**Order:** `check-data` -> `make-lists` -> `tune` -> `test` -> `routing` -> `figures`.

`tune` samples tuning IDs from `train2017`; `test` and `routing` use the disjoint test IDs sampled from `val2017`. Tuning and test caches are resumable, so a rerun after a timeout continues where it stopped.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/zaeem886/domain-adaptive-colorization.git"
BRANCH = "experiment_code"  # change if your E1/E2 branch has a different name
REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


## Dependencies

Kaggle already provides torch, transformers, numpy, Pillow and matplotlib. Only `segment-anything` is installed here, as required by the work plan.


In [ ]:
import importlib.util

if importlib.util.find_spec("segment_anything") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "segment-anything==1.0"],
        check=True,
    )
    print("installed segment-anything==1.0")
else:
    print("segment-anything already installed")


## Dataset paths (edit this cell)

Set `DATASET_ROOT` to the mounted dataset folder. The check at the end fails early with a clear message if a path is wrong.


In [ ]:
DATASET_ROOT = Path("/kaggle/input/coco-stuff-10k")

IMAGES_ROOT = DATASET_ROOT / "images"
LABELS_ROOT = DATASET_ROOT / "stuffthingmaps"
LABEL_NAMES = DATASET_ROOT / "labels.txt"

# Optional: point at an attached SAM checkpoint; "" lets the pipeline download it.
SAM_CHECKPOINT = ""  # e.g. "/kaggle/input/sam-vit-b/sam_vit_b.pth"

# Use a small integer for a smoke run, or None for the full experiment.
LIMIT = None

for label, path in (
    ("images", IMAGES_ROOT),
    ("stuffthingmaps", LABELS_ROOT),
    ("labels.txt", LABEL_NAMES),
):
    print(f"{label:16s} {path}  exists={path.exists()}")

assert IMAGES_ROOT.is_dir(), "Fix IMAGES_ROOT in this cell"
assert LABELS_ROOT.is_dir(), "Fix LABELS_ROOT in this cell"
assert LABEL_NAMES.is_file(), "Fix LABEL_NAMES in this cell"


## Common command builder

Every stage gets the same paths, device and optional `--limit`. Change one variable above rather than editing each command.


In [ ]:
COMMON = [
    "--images-root", str(IMAGES_ROOT),
    "--labels-root", str(LABELS_ROOT),
    "--label-names", str(LABEL_NAMES),
    "--class-groups", "configs/class_groups.json",
    "--cache-dir", "cache",
    "--results-dir", "results",
    "--data-lists-dir", "data_lists",
    "--tuning-ids", "data_lists/e1_tuning_ids.txt",
    "--test-ids", "data_lists/e1_test_ids.txt",
    "--train-split", "train2017",
    "--val-split", "val2017",
]
if SAM_CHECKPOINT:
    COMMON += ["--sam-checkpoint", str(SAM_CHECKPOINT)]
if LIMIT is not None:
    COMMON += ["--limit", str(int(LIMIT))]


def run_e1_e2(*stage_args):
    """Run one experiments/e1_e2.py stage with the shared arguments."""
    command = [sys.executable, "experiments/e1_e2.py", *stage_args, *COMMON]
    print("$", " ".join(shlex.quote(str(part)) for part in command))
    subprocess.run(command, check=True)


## 1. Check label maps and the class offset

Inspect `results/check_data/offset_report.txt` and the rendered label maps before trusting any score. The official Caffe-compatible maps use 0..181 with 255 as void; the runner records the offset it chose.


In [ ]:
run_e1_e2("check-data", "--split", "val2017", "--n-images", "4")


In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import display

check_dir = Path("results/check_data")
report = check_dir / "offset_report.txt"
if report.is_file():
    print(report.read_text())

for path in sorted(check_dir.glob("*_labels.png"))[:4]:
    display(NotebookImage(filename=str(path)))


## 2. Create or reuse the fixed ID lists

The lists are written once and reused on later runs, so tuning and test stay fixed. Pass `--force` to regenerate them (this may break comparability with earlier results).


In [ ]:
run_e1_e2("make-lists", "--tuning-size", "200", "--test-size", "500")


## 3. Tune the five decomposition parameters (train split)

One-pass coordinate sweep over `box_threshold`, `text_threshold`, `min_score`, `nms_iou` and `merge_containment`. Selection is tuning mIoU with pooled coverage as tie-break. DINO outputs and SAM masks are cached, so later settings only re-run cheap stages.


In [ ]:
run_e1_e2("tune")


In [ ]:
import pandas as pd

sweep_path = Path("results/e1/tuning_sweep.csv")
if sweep_path.is_file():
    display(pd.read_csv(sweep_path))

selected_path = Path("results/e1/selected_params.json")
if selected_path.is_file():
    print(json.dumps(json.loads(selected_path.read_text()), indent=2))


## 4. Evaluate E1 on the fixed test IDs (val split)

This writes the decomposition caches that E2 reads. It refuses to run without `results/e1/selected_params.json` unless `--use-defaults` is passed explicitly.


In [ ]:
run_e1_e2("test")


In [ ]:
summary_path = Path("results/e1/summary.json")
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    for key in (
        "n_images", "miou", "domain_miou", "pixel_accuracy",
        "coverage_pooled", "fallback_fraction", "fallback_images",
        "mean_regions", "mean_discarded", "used_defaults",
    ):
        print(f"{key:20s} {summary.get(key)}")
    print("group IoUs:", summary.get("ious"))
    print("domain IoUs:", summary.get("domain_ious"))


## 5. Evaluate E2 routing from the cached test decompositions

No model calls: E2 loads each cached decomposition and compares the assigned domain with the domain of the dominant GT class inside the resolved region mask.


In [ ]:
run_e1_e2("routing", "--region-mask", "resolved")


In [ ]:
routing_path = Path("results/e2/routing_summary.json")
if routing_path.is_file():
    routing = json.loads(routing_path.read_text())
    print(json.dumps(routing, indent=2))


## 6. Figures and notes

All figures are generated from the CSV/JSON files produced above; nothing is hand-edited.


In [ ]:
run_e1_e2("figures")


In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import Markdown, display

figure_dir = Path("results/figures")
for name in (
    "e1_confusion_groups.png",
    "e1_confusion_domains.png",
    "e2_confusion_domains.png",
    "e1_examples_random.png",
    "e1_examples_failures.png",
):
    path = figure_dir / name
    if path.is_file():
        display(NotebookImage(filename=str(path)))

notes_path = Path("results/e1_e2_notes.md")
if notes_path.is_file():
    display(Markdown(notes_path.read_text()))


## 7. Download the results

The archive contains `results/` and the fixed `data_lists/`. Model caches are intentionally excluded (they are large and reproducible).


In [ ]:
staging = Path("/kaggle/working/e1_e2_bundle")
shutil.rmtree(staging, ignore_errors=True)
staging.mkdir(parents=True, exist_ok=True)
if Path("results").is_dir():
    shutil.copytree("results", staging / "results")
if Path("data_lists").is_dir():
    shutil.copytree("data_lists", staging / "data_lists")
archive = shutil.make_archive("/kaggle/working/e1_e2_results", "zip", root_dir=staging)
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
